# Qwen3-VL Embedding Pipeline

Embeds marketing text using **Qwen3-VL-Embedding-2B** and stores vectors in **ChromaDB**.

Model: https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B

Runtime: GPU (T4 or better recommended).

In [1]:
# from sentence_transformers import SentenceTransformer

# model = SentenceTransformer("Qwen/Qwen3-VL-Embedding-2B")

# sentences = [
#     "That is a happy person",
#     "That is a happy dog",
#     "That is a very happy person",
#     "Today is a sunny day"
# ]
# embeddings = model.encode(sentences)

# similarities = model.similarity(embeddings, embeddings)
# print(similarities.shape)
# # [4, 4]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/770 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

torch.Size([4, 4])


In [3]:
from sentence_transformers import SentenceTransformer
import chromadb
from datetime import datetime, timezone

!pip install -q "chromadb==0.5.23" sentence-transformers "transformers>=4.57.0" qwen-vl-utils

In [4]:
model = SentenceTransformer("Qwen/Qwen3-VL-Embedding-2B")
print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Model loaded.


## 2. Embedding helpers

In [5]:
def embed_texts(texts, instruction="Represent the user's input."):
    return model.encode(texts, prompt=instruction, convert_to_numpy=True).tolist()

def embed_query(query):
    return model.encode(
        [query],
        prompt="Retrieve relevant documents for the query.",
        convert_to_numpy=True,
    )[0].tolist()

## 3. Set up ChromaDB

In [6]:
client = chromadb.Client()
collection = client.get_or_create_collection(
    name="arena_embeddings",
    metadata={"hnsw:space": "cosine"},
)
print(f"Collection ready: {collection.name}")

Collection ready: arena_embeddings


## 4. Embed sample marketing documents

In [7]:
sample_docs = [
    "Summer sale campaign targeting young adults on Instagram with 20% discount on apparel.",
    "B2B SaaS email campaign promoting enterprise plan to CTOs and VPs of Engineering.",
    "Facebook retargeting campaign for users who abandoned shopping cart in the last 7 days.",
    "Google Search campaign for high-intent buyers searching for project management software.",
    "LinkedIn sponsored content targeting HR professionals for an HR automation tool.",
]

embeddings = embed_texts(sample_docs)
print(f"Embedded {len(embeddings)} documents — dimension: {len(embeddings[0])}")

Embedded 5 documents — dimension: 2048


## 5. Store in ChromaDB

In [8]:
ts = datetime.now(timezone.utc).isoformat()
ids = [f"doc_{ts}_{i}" for i in range(len(sample_docs))]
metadata = [{"source": "sample", "index": i} for i in range(len(sample_docs))]

collection.add(ids=ids, embeddings=embeddings, documents=sample_docs, metadatas=metadata)
print(f"Stored {collection.count()} documents in ChromaDB")

Stored 5 documents in ChromaDB


## 6. Semantic search

In [9]:
query_text = "email marketing campaign for software companies"
query_embedding = embed_query(query_text)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "distances"],
)

print(f"Query: '{query_text}'\n")
print("Top results:")
for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
    print(f"{i+1}. [score: {1 - dist:.4f}] {doc}")

Query: 'email marketing campaign for software companies'

Top results:
1. [score: 0.7222] B2B SaaS email campaign promoting enterprise plan to CTOs and VPs of Engineering.
2. [score: 0.5976] Google Search campaign for high-intent buyers searching for project management software.
3. [score: 0.5051] LinkedIn sponsored content targeting HR professionals for an HR automation tool.
